LightGBM Approach

In [2]:
# Run this in a cell first if you don't have it installed
!pip install lightgbm

In [2]:
# ==============================================================================
# SECTION 1: IMPORTS & CONFIGURATION
# ==============================================================================
import time
import gc
import re
import polars as pl
import pyarrow.parquet as pq
import numpy as np
import lightgbm as lgb
from sklearn.metrics import ndcg_score

# --- DIRECTORY PATHS ---
DRIVE_PATH = '/content/drive/Shareddrives/CMPE256_FinalProject/board_game_recommendation'
SPLIT_DIR = f"{DRIVE_PATH}/processed_data/final_foundation/splits"
METADATA_PATH = f"{DRIVE_PATH}/processed_data/items_metadata_final.parquet"
MODEL_SAVE_PATH = f"{DRIVE_PATH}/processed_data/lgbm_batch_hybrid_with_val.txt"

# --- EVALUATION SETTINGS ---
K_METRICS = 10          # The '10' in Precision@10, Recall@10, NDCG@10
THRESHOLD = 7.0         # Any rating >= 7.0 is a "Relevant" recommendation
SAMPLE_USERS = 1000     # Number of users to sample for ranking metrics
BATCH_SIZE = 500_000    # Rows to process at a time to prevent RAM crashes

print("✅ Section 1 Complete: Configured.")

✅ Section 1 Complete: Configured.


In [3]:
# ==============================================================================
# SECTION 2: FEATURE CACHING & METADATA SANITIZATION
# ==============================================================================
print("\n🧮 Pre-calculating Collaborative Features & Sanitizing Metadata...")

# 1. Global Mean & User/Item Bias (From Train Set)
train_lazy = pl.scan_parquet(f"{SPLIT_DIR}/train.parquet")
global_mean = train_lazy.select(pl.col("Rating").mean()).collect().item()

user_stats = train_lazy.group_by("user_id").agg([
    pl.col("Rating").mean().alias("user_avg").cast(pl.Float32),
    pl.col("Rating").count().alias("user_count").cast(pl.Float32)
]).collect()

item_stats = train_lazy.group_by("item_id").agg([
    pl.col("Rating").mean().alias("item_avg").cast(pl.Float32),
    pl.col("Rating").count().alias("item_count").cast(pl.Float32)
]).collect()

# 2. Metadata Sanitization (Fixing the JSON Error)
meta_df = pl.read_parquet(METADATA_PATH)
raw_meta_cols = [
    col for col in meta_df.columns
    if meta_df[col].dtype in [pl.Float32, pl.Float64, pl.Int32, pl.Int64, pl.UInt32, pl.UInt64]
    and col not in ['BGGId', 'item_id', 'Title', 'Description']
]

rename_map = {col: re.sub(r'[^A-Za-z0-9_]', '_', col) for col in raw_meta_cols}
meta_df_clean = meta_df.select(["item_id"] + raw_meta_cols).rename(rename_map)
meta_cols = list(rename_map.values())

feature_names = ["user_avg", "user_count", "item_avg", "item_count"] + meta_cols
print(f"✅ Section 2 Complete: Sanitized {len(meta_cols)} features.")


🧮 Pre-calculating Collaborative Features & Sanitizing Metadata...
✅ Section 2 Complete: Sanitized 416 features.


In [4]:
# ==============================================================================
# SECTION 3: THE OUT-OF-CORE BATCH GENERATOR (RAM PROTECTOR)
# ==============================================================================
def process_in_batches(file_path, batch_size=BATCH_SIZE):
    """Uses PyArrow to stream massive files sequentially."""
    parquet_file = pq.ParquetFile(file_path)

    for record_batch in parquet_file.iter_batches(batch_size=batch_size):
        batch = (
            pl.from_arrow(record_batch)
            .join(user_stats, on="user_id", how="left")
            .join(item_stats, on="item_id", how="left")
            .join(meta_df_clean, on="item_id", how="left")
            .with_columns([
                pl.col("user_avg").fill_null(global_mean),
                pl.col("item_avg").fill_null(global_mean),
                pl.col("user_count").fill_null(0.0),
                pl.col("item_count").fill_null(0.0)
            ]).fill_null(0.0)
        )

        X = batch.select(feature_names).to_numpy().astype(np.float32)
        y = batch["Rating"].to_numpy().astype(np.float32)
        yield X, y

        del batch
        gc.collect()

In [7]:
# ==============================================================================
# SECTION 4: INCREMENTAL TRAINING (WITH VALIDATION)
# ==============================================================================
print("\n🧮 Loading Validation Set (Sampled to protect RAM)...")

# OPTIMIZATION 1: Only load a random 150,000 rows of the validation set
# We scan it lazily, shuffle it, and take a small chunk so it easily fits in memory.
val_df = pl.scan_parquet(f"{SPLIT_DIR}/val.parquet").collect().sample(n=150_000, seed=42)

val_batch = (
    val_df
    .join(user_stats, on="user_id", how="left")
    .join(item_stats, on="item_id", how="left")
    .join(meta_df_clean, on="item_id", how="left")
    .with_columns([
        pl.col("user_avg").fill_null(global_mean),
        pl.col("item_avg").fill_null(global_mean),
        pl.col("user_count").fill_null(0.0),
        pl.col("item_count").fill_null(0.0)
    ]).fill_null(0.0)
)

X_val = val_batch.select(feature_names).to_numpy().astype(np.float32)
y_val = val_batch["Rating"].to_numpy().astype(np.float32)

# Create the validation dataset and free the raw arrays immediately
val_data = lgb.Dataset(X_val, label=y_val, feature_name=feature_names, free_raw_data=False)

# OPTIMIZATION 2: Aggressive Garbage Collection
del val_df, val_batch, X_val, y_val
gc.collect()

# OPTIMIZATION 3: Lower Batch Size to 200,000 (Change this here)
SAFE_BATCH_SIZE = 200_000

print(f"\n🚀 Training LightGBM Incrementally ({SAFE_BATCH_SIZE:,} rows per batch)...")
start_time = time.time()

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.1,
    'num_leaves': 63,
    'max_depth': 8,
    'feature_fraction': 0.8,
    'n_jobs': -1,
    'verbose': -1
}

model = None
batch_count = 0

for X_batch, y_batch in process_in_batches(f"{SPLIT_DIR}/train.parquet", batch_size=SAFE_BATCH_SIZE):
    train_data = lgb.Dataset(X_batch, label=y_batch, feature_name=feature_names, free_raw_data=True)

    model = lgb.train(
        params,
        train_data,
        num_boost_round=15,
        valid_sets=[val_data],
        valid_names=['validation'],
        init_model=model,
        keep_training_booster=True
    )

    batch_count += 1
    print(f"  Processed {batch_count * SAFE_BATCH_SIZE:,} rows...")

    # Force RAM cleanup after every single batch
    del X_batch, y_batch, train_data
    gc.collect()


🧮 Loading Validation Set (Sampled to protect RAM)...

🚀 Training LightGBM Incrementally (200,000 rows per batch)...
  Processed 200,000 rows...
  Processed 400,000 rows...
  Processed 600,000 rows...
  Processed 800,000 rows...
  Processed 1,000,000 rows...
  Processed 1,200,000 rows...
  Processed 1,400,000 rows...
  Processed 1,600,000 rows...
  Processed 1,800,000 rows...
  Processed 2,000,000 rows...
  Processed 2,200,000 rows...
  Processed 2,400,000 rows...
  Processed 2,600,000 rows...
  Processed 2,800,000 rows...
  Processed 3,000,000 rows...
  Processed 3,200,000 rows...
  Processed 3,400,000 rows...
  Processed 3,600,000 rows...
  Processed 3,800,000 rows...
  Processed 4,000,000 rows...
  Processed 4,200,000 rows...
  Processed 4,400,000 rows...
  Processed 4,600,000 rows...
  Processed 4,800,000 rows...
  Processed 5,000,000 rows...
  Processed 5,200,000 rows...
  Processed 5,400,000 rows...
  Processed 5,600,000 rows...
  Processed 5,800,000 rows...
  Processed 6,000,000

In [8]:
# Save Model
model.save_model(MODEL_SAVE_PATH)
print(f"✅ Training completed in {time.time() - start_time:.2f}s. Model saved.")

✅ Training completed in 3064.75s. Model saved.


In [5]:
# Load Model
"""
When you come back to Colab later, you don't need to run the training cell.
Just run your setup/imports, and then load the model like this:
"""
print("Loading saved model...")
MODEL_PATH = f"{DRIVE_PATH}/processed_data/lgbm_batch_hybrid_with_val.txt"
model = lgb.Booster(model_file=MODEL_PATH)

# You can then use model.predict() exactly as before!


Loading saved model...


In [6]:
# ==============================================================================
# SECTION 5 & 6: FULL TEST SET EVALUATION (RAM-Optimized Baseline Match)
# ==============================================================================
print("\n🚀 Generating Predictions for the Entire Test Set (In Batches)...")
start_time = time.time()

# 1. Generate predictions chunk-by-chunk to completely protect RAM
all_preds = []
batch_count = 0

for X_test, y_test in process_in_batches(f"{SPLIT_DIR}/test.parquet", batch_size=200_000):
    # Predict the batch and immediately save just the 1D array of scores
    batch_preds = np.clip(model.predict(X_test), 1, 10)
    all_preds.append(batch_preds)

    batch_count += 1
    print(f"  Predicted {batch_count * 200_000:,} test rows...")

    # Dump the massive feature matrix from RAM
    del X_test, y_test, batch_preds
    gc.collect()

# 2. Combine all batch predictions into one single array
final_predictions = np.concatenate(all_preds)
print("✅ All test predictions generated.")

# 3. Load the lightweight base test file (NO metadata joins)
test_df = pl.read_parquet(f"{SPLIT_DIR}/test.parquet")

# Snap the predictions onto the dataframe
results_df = test_df.with_columns(pl.Series("predicted", final_predictions))

# Clean up memory
del all_preds, final_predictions, test_df
gc.collect()


# 4. Adapt the baseline Evaluation Framework Function
def calculate_metrics_polars_lgbm(df, k=10, threshold=7.0):
    """
    Exact replica of the baseline evaluation framework.
    Calculates RMSE, Precision@K, Recall@K, and NDCG@K using the full test set.
    """
    # Ranking Logic
    sorted_df = df.sort(["user_id", "predicted"], descending=[False, True])
    ranked_df = sorted_df.with_columns(
        pl.col("predicted").rank(method="ordinal", descending=True).over("user_id").alias("rank")
    )
    top_k = ranked_df.filter(pl.col("rank") <= k)

    # Precision & Recall Components
    user_metrics = ranked_df.group_by("user_id").agg([
        pl.col("Rating").filter(pl.col("Rating") >= threshold).count().alias("n_rel"),
        pl.col("Rating")
            .filter((pl.col("rank") <= k) & (pl.col("Rating") >= threshold))
            .count()
            .alias("n_rel_and_rec_k")
    ])

    user_metrics = user_metrics.with_columns([
        (pl.col("n_rel_and_rec_k") / k).alias("precision"),
        (pl.when(pl.col("n_rel") > 0)
         .then(pl.col("n_rel_and_rec_k") / pl.col("n_rel"))
         .otherwise(0.0)).alias("recall")
    ])

    # NDCG Components
    top_k_dcg = top_k.with_columns(
        (pl.col("Rating") / np.log2(pl.col("rank") + 1)).alias("dcg_item")
    ).group_by("user_id").agg(pl.col("dcg_item").sum().alias("dcg"))

    ideal_top_k = ranked_df.sort(["user_id", "Rating"], descending=[False, True]) \
        .with_columns(pl.col("Rating").rank(method="ordinal", descending=True).over("user_id").alias("ideal_rank")) \
        .filter(pl.col("ideal_rank") <= k)

    top_k_idcg = ideal_top_k.with_columns(
        (pl.col("Rating") / np.log2(pl.col("ideal_rank") + 1)).alias("idcg_item")
    ).group_by("user_id").agg(pl.col("idcg_item").sum().alias("idcg"))

    ndcg_df = top_k_dcg.join(top_k_idcg, on="user_id").with_columns(
        (pl.when(pl.col("idcg") > 0)
         .then(pl.col("dcg") / pl.col("idcg"))
         .otherwise(0.0)).alias("ndcg")
    )

    # Final Aggregation
    return {
        "RMSE": np.sqrt(((df["Rating"] - df["predicted"])**2).mean()),
        "Precision@10": user_metrics["precision"].mean(),
        "Recall@10": user_metrics["recall"].mean(),
        "NDCG@10": ndcg_df["ndcg"].mean()
    }


# 5. Execute the Baseline Framework
print("\n🧮 Calculating final metrics...")
metrics = calculate_metrics_polars_lgbm(results_df, k=K_METRICS, threshold=THRESHOLD)

# 6. Final Output Report
print(f"✅ Predictions and Metrics Calculated in {time.time() - start_time:.2f}s")
print("\n" + "="*50)
print("  FINAL LIGHTGBM RESULTS (FULL TEST SET EVALUATION)")
print("="*50)
for m_name, m_val in metrics.items():
    print(f"{m_name:12}: {m_val:.4f}")
print("="*50)


🚀 Generating Predictions for the Entire Test Set (In Batches)...
  Predicted 200,000 test rows...
  Predicted 400,000 test rows...
  Predicted 600,000 test rows...
  Predicted 800,000 test rows...
  Predicted 1,000,000 test rows...
  Predicted 1,200,000 test rows...
  Predicted 1,400,000 test rows...
  Predicted 1,600,000 test rows...
  Predicted 1,800,000 test rows...
  Predicted 2,000,000 test rows...
  Predicted 2,200,000 test rows...
✅ All test predictions generated.

🧮 Calculating final metrics...
✅ Predictions and Metrics Calculated in 271.29s

  FINAL LIGHTGBM RESULTS (FULL TEST SET EVALUATION)
RMSE        : 1.2643
Precision@10: 0.2672
Recall@10   : 0.9059
NDCG@10     : 0.9846


In [9]:
# ==============================================================================
# SECTION 5: GLOBAL RMSE EVALUATION (ON TEST SET)
# ==============================================================================
print("\n🚀 Calculating Global RMSE (Streaming all test data)...")
start_time = time.time()
total_se = 0.0
total_n = 0

for X_test, y_test in process_in_batches(f"{SPLIT_DIR}/test.parquet"):
    preds = np.clip(model.predict(X_test), 1, 10)
    total_se += np.sum((y_test - preds) ** 2)
    total_n += len(y_test)

    del X_test, y_test
    gc.collect()

final_rmse = np.sqrt(total_se / total_n)
print(f"✅ RMSE Calculated in {time.time() - start_time:.2f}s")


🚀 Calculating Global RMSE (Streaming all test data)...
✅ RMSE Calculated in 331.89s


In [10]:
# ==============================================================================
# SECTION 6: RANKING METRICS (P@10, R@10, NDCG@10)
# ==============================================================================
print(f"\n🚀 Calculating Ranking Metrics (Sampled {SAMPLE_USERS} Users)...")
start_time = time.time()

test_df = pl.read_parquet(f"{SPLIT_DIR}/test.parquet")
user_counts = test_df.group_by("user_id").agg(pl.count().alias("count"))
valid_users = user_counts.filter(pl.col("count") >= K_METRICS)["user_id"].to_list()

sample_users = np.random.choice(valid_users, size=min(SAMPLE_USERS, len(valid_users)), replace=False)
eval_df = test_df.filter(pl.col("user_id").is_in(sample_users))

eval_features = (
    eval_df
    .join(user_stats, on="user_id", how="left")
    .join(item_stats, on="item_id", how="left")
    .join(meta_df_clean, on="item_id", how="left")
    .with_columns([
        pl.col("user_avg").fill_null(global_mean),
        pl.col("item_avg").fill_null(global_mean),
        pl.col("user_count").fill_null(0.0),
        pl.col("item_count").fill_null(0.0)
    ]).fill_null(0.0)
)

X_eval = eval_features.select(feature_names).to_numpy().astype(np.float32)
eval_features = eval_features.with_columns(pl.Series("predicted", model.predict(X_eval)))

ndcg_list, precision_list, recall_list = [], [], []

for _, group in eval_features.group_by("user_id"):
    y_true = group["Rating"].to_numpy()
    y_pred = group["predicted"].to_numpy()

    ndcg_list.append(ndcg_score([y_true], [y_pred], k=K_METRICS))

    true_relevant_idx = np.where(y_true >= THRESHOLD)[0]
    num_true_relevant = len(true_relevant_idx)

    top_k_pred_idx = np.argsort(y_pred)[-K_METRICS:][::-1]
    hits = len(set(top_k_pred_idx).intersection(set(true_relevant_idx)))

    precision_list.append(hits / K_METRICS)
    if num_true_relevant > 0:
        recall_list.append(hits / num_true_relevant)

final_ndcg = np.mean(ndcg_list)
final_precision = np.mean(precision_list)
final_recall = np.mean(recall_list)


🚀 Calculating Ranking Metrics (Sampled 1000 Users)...


/tmp/ipykernel_16554/2525770706.py:8: DeprecationWarning: `pl.count()` is deprecated. Please use `pl.len()` instead.
(Deprecated in version 0.20.5)
  user_counts = test_df.group_by("user_id").agg(pl.count().alias("count"))


In [11]:
# --- FINAL OUTPUT REPORT ---
print("\n" + "="*45)
print("     LIGHTGBM HYBRID METRICS REPORT")
print("="*45)
print(f"  RMSE:          {final_rmse:.4f}  (Global Test Set)")
print(f"  NDCG@{K_METRICS}:       {final_ndcg:.4f}  (Sampled)")
print(f"  Precision@{K_METRICS}:  {final_precision:.4f}  (Sampled, Threshold: {THRESHOLD})")
print(f"  Recall@{K_METRICS}:     {final_recall:.4f}  (Sampled, Threshold: {THRESHOLD})")
print("="*45)


     LIGHTGBM HYBRID METRICS REPORT
  RMSE:          1.2643  (Global Test Set)
  NDCG@10:       0.9332  (Sampled)
  Precision@10:  0.7768  (Sampled, Threshold: 7.0)
  Recall@10:     0.6800  (Sampled, Threshold: 7.0)
